Download packages

In [ ]:
#Need to install
#!pip install pandas numpy sklearn.metrics sklearn.model_selection emoji re datasets transformers warnings


In [1]:
# For data manipulation and analysis
import pandas as pd
import numpy as np

# For machine learning tools and evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


from sklearn.model_selection import train_test_split

import emoji
import re
import datasets
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import warnings
from sklearn.utils.class_weight import compute_class_weight
import torch 

c:\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Get tokenizer

In [2]:
tokenizer = AutoTokenizer.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Get BERT model

In [3]:
model = AutoModelForSequenceClassification.from_pretrained("DTAI-KULeuven/robbert-2022-dutch-base")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at DTAI-KULeuven/robbert-2022-dutch-base and are newly initialized: ['classifier.dense.bias', 'classifier.out_proj.bias', 'classifier.dense.weight', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Prepare data

In [ ]:
# Read data 
data =  pd.read_csv("data.csv")
# Classification only works when outcome variable is called labels
data[['labels']] = data[['label_mis']]
# Convert emoji to descriptions (in English) using emoji package
def no_emoji(text):
    text = emoji.demojize(text) 
    return text 
data['text'] = data['text'].apply(no_emoji)
#Remove urls, &gt, &lt and &amp, and [numbers] from text
#Removed [numbers] because there were meaningless numbers between brackets in the Tweets
def clean_text(text):
    text = re.sub(r'https?://\S+|www\.\S+|\r|\n|&gt.?| &lt.?|&amp.?|\[\d*\]', '', text)
    return text 
data['text'] = data['text'].apply(clean_text) 

#Display cleaned text
pd.set_option('display.max_rows', None)
pd.set_option('max_colwidth', None)

#One dataset with only text and labels to train BERT model
#One dataset with also year so I can later split dataset and evaluate metrics per year
data_inf = data[['text', 'labels', 'id', 'year']]
data = data[['text', 'labels']]



Set values of variables important for training

In [ ]:
max_length = 512
# set random seed for reproducibility
SEED_GLOBAL = 42
np.random.seed(SEED_GLOBAL)
#Set training directory 
training_directory = ""

Split data into training and test set

In [4]:
# Train and test set for training model
df_train, df_test = train_test_split(data, random_state=42, test_size=0.25)

#Create identical test with text and labels plus year so performance metrics can be split by year
df_train_inf, df_test_inf = train_test_split(data_inf, random_state=42, test_size=0.25)

Tokenize data (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [5]:
# convert pandas dataframes to Hugging Face dataset object to facilitate pre-processing
dataset = datasets.DatasetDict({
    "train": datasets.Dataset.from_pandas(df_train),
    "test": datasets.Dataset.from_pandas(df_test)
})

# tokenize
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, padding = 'max_length', max_length=512)  # max_length can be reduced to e.g. 256 to increase speed, but long texts will be cut off

dataset = dataset.map(tokenize, batched=True)

Map: 100%|██████████| 849/849 [00:00<00:00, 3960.48 examples/s]


Set weights 

In [9]:
# Define real-world and dataset distributions
real_dist = np.array([0.89, 0.11])
current_dist = np.array([0.80, 0.20])

# Compute class weights
class_weights = real_dist / current_dist
print("Computed Class Weights:", class_weights)

# Convert to PyTorch tensor
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

Computed Class Weights: [1.1125 0.55  ]


Set training arguments (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [12]:
train_args = TrainingArguments(
    num_train_epochs=3,  
    learning_rate=2e-5,
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=64,   
    warmup_ratio=0.06, 
    weight_decay=0.1,
    seed=SEED_GLOBAL,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    evaluation_strategy="epoch", 
    save_strategy = "epoch",
    report_to="all",
    output_dir=f'{training_directory}',
    logging_dir=f'{training_directory}',
)


Set evaluation metrics (from https://github.com/MoritzLaurer/summer-school-transformers-2023/blob/main/3_tune_bert.ipynb)

In [13]:
# Function to calculate metrics
# documentation on all metrics: https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics

def compute_metrics_standard(eval_pred):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")

        labels = eval_pred.label_ids
        pred_logits = eval_pred.predictions
        preds_max = np.argmax(pred_logits, axis=1) 

        # metrics
        precision_mis, recall_mis, f1_mis, _ = precision_recall_fscore_support(labels, preds_max, average= 'binary') 
        precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(labels, preds_max, average='macro')  
        precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(labels, preds_max, average='micro')  
        acc_balanced = balanced_accuracy_score(labels, preds_max)
        acc_not_balanced = accuracy_score(labels, preds_max)

        metrics = {
            'accuracy': acc_not_balanced,
            'f1_macro': f1_macro,
            'accuracy_balanced': acc_balanced,
            'f1_micro': f1_micro,
            'precision_macro': precision_macro,
            'recall_macro': recall_macro,
            'precision_micro': precision_micro,
            'recall_micro': recall_micro,
            'precision_misinformation': precision_mis,
            'recall_misinformation': recall_mis,
            'f1_misinformation': f1_mis,
        }

        return metrics

Use weighted trainer

In [14]:
from torch import nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

Set BERT model

In [15]:
trainer = WeightedTrainer(
    model=model,                         
    args=train_args,             
    train_dataset=dataset["train"],        
    eval_dataset=dataset["test"],           
    compute_metrics=compute_metrics_standard     
)

Train BERT model 

In [ ]:
trainer.train()

Load original model

In [ ]:
model_path = ""
model_org = AutoModelForSequenceClassification.from_pretrained(model_path)

Load model with balanced classes

In [ ]:
model_path = ""
model_bal = AutoModelForSequenceClassification.from_pretrained(model_path)

Inference original model with original test set

In [10]:
df_inference = df_test[["text", "labels"]].copy(deep=True)
text_lst = df_inference["text"].tolist()
from transformers import pipeline

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model_org,  
    tokenizer=tokenizer,
    framework="pt"
)
#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.4.0+cu121 with CUDA 1201 (you have 2.4.0+cpu)
    Python  3.11.9 (you have 3.11.2)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


[{'label': 'LABEL_0', 'score': 0.991233766078949}, {'label': 'LABEL_0', 'score': 0.9372941255569458}, {'label': 'LABEL_0', 'score': 0.9917673468589783}, {'label': 'LABEL_0', 'score': 0.9755799174308777}, {'label': 'LABEL_0', 'score': 0.8603397607803345}, {'label': 'LABEL_0', 'score': 0.9883157014846802}, {'label': 'LABEL_1', 'score': 0.8581365346908569}, {'label': 'LABEL_0', 'score': 0.984526515007019}, {'label': 'LABEL_0', 'score': 0.9604642391204834}, {'label': 'LABEL_0', 'score': 0.9751541018486023}, {'label': 'LABEL_0', 'score': 0.9871692657470703}, {'label': 'LABEL_0', 'score': 0.9015777707099915}, {'label': 'LABEL_0', 'score': 0.9756954312324524}, {'label': 'LABEL_0', 'score': 0.5768104195594788}, {'label': 'LABEL_0', 'score': 0.9920639395713806}, {'label': 'LABEL_1', 'score': 0.830923318862915}, {'label': 'LABEL_0', 'score': 0.9666747450828552}, {'label': 'LABEL_1', 'score': 0.8869199752807617}, {'label': 'LABEL_0', 'score': 0.9725577235221863}, {'label': 'LABEL_0', 'score': 0.9

In [ ]:
# add inference data to original dataframe
#copy test set so original does not get replaced 
df_inference["label_text_pred"] = df_output["label"].tolist()
df_inference["label_text_pred_probability"] = df_output["score"].round(2).tolist()

#Convert label_1 and label_0 to 1 and 0
df_inference['label_pred'] = df_inference['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)
#Print df_inference
df_inference

Calculate f1, recall and precision original model on original test set

In [ ]:
# Calculate precision and recall overall (to test if this gets me the same results as main analysis)
prec_r_f = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='macro')
prec_r_f_m = precision_recall_fscore_support(df_inference['labels'], df_inference['label_pred'], average= 'binary')
print(f'Precision_rec_f: {prec_r_f}')
print(f'Precision_rec_f misinformation: {prec_r_f_m}')

Precision_rec_f: (0.7850869280898369, 0.7125832324455206, 0.7381765761975227, None)
Precision_rec_f misinformation: (0.6967213114754098, 0.480225988700565, 0.568561872909699, None)


Inference balanced model with original test set

In [11]:
df_inference = df_test[["text", "labels"]].copy(deep=True)
text_lst = df_inference["text"].tolist()

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model_bal,  
    tokenizer=tokenizer,
    framework="pt"
)
#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

[{'label': 'LABEL_0', 'score': 0.9954912066459656}, {'label': 'LABEL_0', 'score': 0.9804994463920593}, {'label': 'LABEL_0', 'score': 0.9960538148880005}, {'label': 'LABEL_0', 'score': 0.9875409007072449}, {'label': 'LABEL_0', 'score': 0.9356276392936707}, {'label': 'LABEL_0', 'score': 0.9928911924362183}, {'label': 'LABEL_1', 'score': 0.8370327949523926}, {'label': 'LABEL_0', 'score': 0.9845128655433655}, {'label': 'LABEL_0', 'score': 0.9791991114616394}, {'label': 'LABEL_0', 'score': 0.9860629439353943}, {'label': 'LABEL_0', 'score': 0.9940999150276184}, {'label': 'LABEL_0', 'score': 0.954319417476654}, {'label': 'LABEL_0', 'score': 0.9927412271499634}, {'label': 'LABEL_0', 'score': 0.5472304821014404}, {'label': 'LABEL_0', 'score': 0.9939994812011719}, {'label': 'LABEL_0', 'score': 0.5939300656318665}, {'label': 'LABEL_0', 'score': 0.9868495464324951}, {'label': 'LABEL_1', 'score': 0.6774288415908813}, {'label': 'LABEL_0', 'score': 0.974290132522583}, {'label': 'LABEL_0', 'score': 0.

Calculate f1, recall and precision balanced model on original test set

In [ ]:
# add inference data to original dataframe
df_inference["label_text_pred"] = df_output["label"].tolist()
df_inference["label_text_pred_probability"] = df_output["score"].round(2).tolist()

#Convert label_1 and label_0 to 1 and 0
df_inference['label_pred'] = df_inference['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)
#Print df_inference
df_inference

In [13]:
# Calculate precision and recall overall, for misinformation (to test if this gets me the same results as main analysis) and for accurate information
prec_r_f = precision_recall_fscore_support(y_true=df_inference['labels'], y_pred=df_inference['label_pred'], average='macro')
prec_r_f_m = precision_recall_fscore_support(df_inference['labels'], df_inference['label_pred'], average= 'binary')
prec_r_f_a = precision_recall_fscore_support(df_inference['labels'], df_inference['label_pred'], average= 'binary', pos_label=0)
print(f'Precision_rec_f: {prec_r_f}')
print(f'Precision_rec_f misinformation: {prec_r_f_m}')
print(f'Precision_rec_f accurate information: {prec_r_f_a}')

Precision_rec_f: (0.7895166971959984, 0.6727451573849879, 0.7036204769896333, None)
Precision_rec_f misinformation: (0.723404255319149, 0.384180790960452, 0.5018450184501846, None)
Precision_rec_f accurate information: (0.8556291390728477, 0.9613095238095238, 0.9053959355290818, None)


Make real-world distribution test set

In [ ]:
# need 83 tweets containing misinformation to have 11% misinformation, so randomly remove 94 misinformation tweets from dataset 
# Filtering label 1 and dropping 94 random cases
df_label_1 = df_test[df_test["labels"] == 1].sample(n=94, random_state=42)
df_inf_real = df_test.drop(df_label_1.index)

# Display the result
print(df_inf_real)


In [15]:
#Check if sampling went well
df_inf_real['labels'].value_counts()

0    672
1     83
Name: labels, dtype: int64

Inference balanced model with real-world distribution test set

In [24]:
text_lst = df_inf_real["text"].tolist()

# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model_bal,  
    tokenizer=tokenizer,
    framework="pt"
)
#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

[{'label': 'LABEL_0', 'score': 0.9954912066459656}, {'label': 'LABEL_0', 'score': 0.9804994463920593}, {'label': 'LABEL_0', 'score': 0.9960538148880005}, {'label': 'LABEL_0', 'score': 0.9875409007072449}, {'label': 'LABEL_0', 'score': 0.9356276392936707}, {'label': 'LABEL_0', 'score': 0.9928911924362183}, {'label': 'LABEL_0', 'score': 0.9845128655433655}, {'label': 'LABEL_0', 'score': 0.9791991114616394}, {'label': 'LABEL_0', 'score': 0.9860629439353943}, {'label': 'LABEL_0', 'score': 0.9940999150276184}, {'label': 'LABEL_0', 'score': 0.954319417476654}, {'label': 'LABEL_0', 'score': 0.9927412271499634}, {'label': 'LABEL_0', 'score': 0.9939994812011719}, {'label': 'LABEL_0', 'score': 0.5939300656318665}, {'label': 'LABEL_0', 'score': 0.9868495464324951}, {'label': 'LABEL_0', 'score': 0.974290132522583}, {'label': 'LABEL_0', 'score': 0.9920672178268433}, {'label': 'LABEL_0', 'score': 0.8251925706863403}, {'label': 'LABEL_0', 'score': 0.9972425699234009}, {'label': 'LABEL_0', 'score': 0.

In [ ]:
# add inference data to original dataframe
#copy test set so original does not get replaced 
df_inf_real["label_text_pred"] = df_output["label"].tolist()
df_inf_real["label_text_pred_probability"] = df_output["score"].round(2).tolist()

#Convert label_1 and label_0 to 1 and 0
df_inf_real['label_pred'] = df_inf_real['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)
#Print df_inference
df_inf_real

Calculate f1, recall and precision model with balanced classes with test set real-world distribution

In [26]:
# Calculate precision and recall overall,  
prec_r_f = precision_recall_fscore_support(y_true=df_inf_real['labels'], y_pred=df_inf_real['label_pred'], average='macro')
prec_r_f_m = precision_recall_fscore_support(df_inf_real['labels'], df_inf_real['label_pred'], average= 'binary')
prec_r_f_a = precision_recall_fscore_support(df_inf_real['labels'], df_inf_real['label_pred'], average= 'binary', pos_label=0)
accuracy = accuracy_score(df_inf_real['labels'], df_inf_real['label_pred'])
print(f'Precision_rec_f: {prec_r_f}')
print(f'Precision_rec_f misinformation: {prec_r_f_m}')
print(f'Precision_rec_f accurate information: {prec_r_f_a}')
print(f'Accuracy: {accuracy}')

Precision_rec_f: (0.7346805408937818, 0.6674017498565692, 0.6929614181438999, None)
Precision_rec_f misinformation: (0.543859649122807, 0.37349397590361444, 0.44285714285714284, None)
Precision_rec_f accurate information: (0.9255014326647565, 0.9613095238095238, 0.9430656934306569, None)
Accuracy: 0.8966887417218543


Inference original model with test set real-world distribution

In [19]:
text_lst = df_inf_real["text"].tolist()
# documentation: https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.ZeroShotClassificationPipeline
pipe_classifier = pipeline(
    "text-classification",
    model=model_org,  
    tokenizer=tokenizer,
    framework="pt"
)
#inference
pipe_output = pipe_classifier(
    text_lst, 
    batch_size=64 
)
print(pipe_output)

df_output = pd.DataFrame(pipe_output)

[{'label': 'LABEL_0', 'score': 0.991233766078949}, {'label': 'LABEL_0', 'score': 0.9372941255569458}, {'label': 'LABEL_0', 'score': 0.9917673468589783}, {'label': 'LABEL_0', 'score': 0.9755799174308777}, {'label': 'LABEL_0', 'score': 0.8603397607803345}, {'label': 'LABEL_0', 'score': 0.9883157014846802}, {'label': 'LABEL_0', 'score': 0.984526515007019}, {'label': 'LABEL_0', 'score': 0.9604642391204834}, {'label': 'LABEL_0', 'score': 0.9751541018486023}, {'label': 'LABEL_0', 'score': 0.9871692657470703}, {'label': 'LABEL_0', 'score': 0.9015777707099915}, {'label': 'LABEL_0', 'score': 0.9756954312324524}, {'label': 'LABEL_0', 'score': 0.9920639395713806}, {'label': 'LABEL_1', 'score': 0.830923318862915}, {'label': 'LABEL_0', 'score': 0.9666747450828552}, {'label': 'LABEL_0', 'score': 0.9725577235221863}, {'label': 'LABEL_0', 'score': 0.9660543203353882}, {'label': 'LABEL_0', 'score': 0.7588967680931091}, {'label': 'LABEL_0', 'score': 0.9951169490814209}, {'label': 'LABEL_0', 'score': 0.5

In [ ]:
# add inference data to original dataframe
df_inf_real["label_text_pred"] = df_output["label"].tolist()
df_inf_real["label_text_pred_probability"] = df_output["score"].round(2).tolist()

#Convert label_1 and label_0 to 1 and 0
df_inf_real['label_pred'] = df_inf_real['label_text_pred'].apply(lambda x: 1 if x == 'LABEL_1' else 0)
#Print df_inference
df_inf_real

Calculate f1, recall and precision original model with test set real-world distribution

In [23]:
# Calculate precision and recall overall, for misinformation and accurate information 
prec_r_f = precision_recall_fscore_support(y_true=df_inf_real['labels'], y_pred=df_inf_real['label_pred'], average='macro')
prec_r_f_m = precision_recall_fscore_support(df_inf_real['labels'], df_inf_real['label_pred'], average= 'binary')
prec_r_f_a = precision_recall_fscore_support(df_inf_real['labels'], df_inf_real['label_pred'], average= 'binary', pos_label=0)
accuracy = accuracy_score(df_inf_real['labels'], df_inf_real['label_pred'])
print(f'Precision_rec_f: {prec_r_f}')
print(f'Precision_rec_f misinformation: {prec_r_f_m}')
print(f'Precision_rec_f accurate information: {prec_r_f_a}')
print(f'Accuracy: {accuracy}')

Precision_rec_f: (0.731801310457145, 0.719458189902467, 0.725377436242167, None)
Precision_rec_f misinformation: (0.5256410256410257, 0.4939759036144578, 0.5093167701863354, None)
Precision_rec_f accurate information: (0.9379615952732644, 0.9449404761904762, 0.9414381022979985, None)
Accuracy: 0.895364238410596
